# Tutorial: Running AblativeDrumstick

This notebook walks through a basic vertical-fall simulation for `AblativeDrumstick`.

Audience:
- Someone who wants to run the model and inspect the main numerical outputs.

Prerequisites:
- A Python kernel with `jax`, `diffrax`, and `numpy` available. Locally, use the `cock` conda environment.
- Run this notebook from inside the repository, or let the setup cell find the repository root.

Learning goals:
- Run the default simulation.
- Inspect trajectory, speed, atmosphere density, drag power, and energy.
- Change a parameter and compare the result.


## Outline

1. Make the local `src/` package importable.
2. Run the default simulation.
3. Inspect the time series and summary metrics.
4. Change one parameter and compare outcomes.
5. Try a small exercise.

## 1. Setup

The project is currently used directly from source, not installed as a package. This cell finds the repository root and adds `src/` to `sys.path`.

In [ ]:
from pathlib import Path
import sys


def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in (start, *start.parents):
        if (path / "src" / "chicken_from_space").exists():
            return path
    raise RuntimeError("Could not find the AblativeDrumstick repository root.")


REPO_ROOT = find_repo_root()
SRC_ROOT = REPO_ROOT / "src"
if str(SRC_ROOT) not in sys.path:
    sys.path.insert(0, str(SRC_ROOT))

print(f"Repository root: {REPO_ROOT}")

Import the small set of objects needed for a basic run.

In [ ]:
from dataclasses import replace

import jax
import jax.numpy as jnp
import numpy as np

from chicken_from_space.config import Config
from chicken_from_space.simulation import (
    format_summary,
    run_simulation,
    summarize_simulation,
)

print(f"JAX backend: {jax.default_backend()}")

## 2. Run The Default Simulation

The default configuration drops the equivalent spherical chicken from the configured initial altitude and stops when it reaches the ground.

In [ ]:
cfg = Config()
result = run_simulation(cfg)
summary = summarize_simulation(result)

print(format_summary(summary))

## 3. Inspect The Time Series

`SimulationResult` stores the full time history as JAX arrays. For quick notebook inspection, convert them to NumPy arrays and print compact summaries.

In [ ]:
series = result.as_numpy_dict()

for name in [
    "time_s",
    "altitude_m",
    "velocity_m_per_s",
    "density_kg_per_m3",
    "drag_power_w",
    "cumulative_drag_energy_j",
]:
    values = series[name]
    print(
        f"{name:28s} shape={values.shape!s:10s} "
        f"first={values[0]:12.4g} last={values[-1]:12.4g}"
    )

A few sampled rows are often easier to read than the full arrays.

In [ ]:
idx = np.linspace(0, len(series["time_s"]) - 1, 6, dtype=int)
rows = []

for i in idx:
    rows.append(
        {
            "time_s": round(float(series["time_s"][i]), 2),
            "altitude_m": round(float(series["altitude_m"][i]), 1),
            "velocity_m_per_s": round(float(series["velocity_m_per_s"][i]), 2),
            "density_kg_per_m3": float(f"{series['density_kg_per_m3'][i]:.3e}"),
            "drag_power_w": round(float(series["drag_power_w"][i]), 2),
        }
    )

rows

## 4. Change One Parameter

The configuration dataclasses are frozen, so create modified copies with `dataclasses.replace`. Here we compare the default chicken with a heavier one.

In [ ]:
heavy_cfg = replace(
    cfg,
    chicken=replace(cfg.chicken, mass_kg=2.0),
)

heavy_result = run_simulation(heavy_cfg)
heavy_summary = summarize_simulation(heavy_result)

comparison = [
    {
        "case": "default",
        "fall_time_s": round(summary.total_fall_time_s, 2),
        "max_speed_m_per_s": round(summary.max_speed_m_per_s, 2),
        "drag_energy_j": round(summary.total_drag_energy_j, 2),
    },
    {
        "case": "heavier",
        "fall_time_s": round(heavy_summary.total_fall_time_s, 2),
        "max_speed_m_per_s": round(heavy_summary.max_speed_m_per_s, 2),
        "drag_energy_j": round(heavy_summary.total_drag_energy_j, 2),
    },
]

comparison

## 5. Exercise

Try changing the drag coefficient. Before running the cell, predict what should happen to fall time, maximum speed, and dissipated energy.

In [ ]:
exercise_cfg = replace(
    cfg,
    chicken=replace(cfg.chicken, drag_coefficient=0.8),
)

exercise_result = run_simulation(exercise_cfg)
exercise_summary = summarize_simulation(exercise_result)

print(format_summary(exercise_summary))

## Pitfalls And Extensions

- If imports fail, make sure the setup cell ran and that the notebook is using an environment with JAX and Diffrax.
- The current atmosphere model is compact by design: temperature is interpolated from nodes, pressure comes from hydrostatic quadrature, and density follows the ideal gas law.
- A useful next step is plotting altitude, speed, and drag power once plotting dependencies are added.